In [2]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [3]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


In [4]:
from pyspark.sql.functions import *

## Actors and Directors Cooperation

You are given a DataFrame **`actor_director`** with columns:
- `actor_id` (int): Actor identifier
- `director_id` (int): Director identifier
- `timestamp` (int): Timestamp of the cooperation

**Task:** Find all actor-director pairs where they have cooperated **at least 3 times**.

Return `actor_id`, `director_id`.

### Example

| actor_id | director_id | timestamp |
|----------|-------------|-----------|
| 1        | 1           | 0         |
| 1        | 1           | 1         |
| 1        | 1           | 2         |
| 1        | 2           | 3         |
| 1        | 2           | 4         |
| 2        | 1           | 5         |
| 2        | 1           | 6         |

Expected output:

| actor_id | director_id |
|----------|-------------|
| 1        | 1           |

In [5]:
data = [
    (1, 1, 0),
    (1, 1, 1),
    (1, 1, 2),
    (1, 2, 3),
    (1, 2, 4),
    (2, 1, 5),
    (2, 1, 6)
]

columns = ["actor_id", "director_id", "timestamp"]

actor_director = spark.createDataFrame(data, columns)

print("Original DataFrame:")
actor_director.show()

Original DataFrame:
+--------+-----------+---------+
|actor_id|director_id|timestamp|
+--------+-----------+---------+
|       1|          1|        0|
|       1|          1|        1|
|       1|          1|        2|
|       1|          2|        3|
|       1|          2|        4|
|       2|          1|        5|
|       2|          1|        6|
+--------+-----------+---------+



# Using Spark SQL

In [6]:
actor_director.createOrReplaceTempView("data")

In [9]:
spark.sql(
    """
    SELECT actor_id,director_id 
    from data 
    group by actor_id,director_id 
    having count(*) >= 3
    
    """
).show()

+--------+-----------+
|actor_id|director_id|
+--------+-----------+
|       1|          1|
+--------+-----------+



# Using Pyspark

In [12]:
actor_director \
    .groupBy("actor_id", "director_id") \
    .count() \
    .filter(col("count") >= 3) \
    .select("actor_id", "director_id") \
    .show()

+--------+-----------+
|actor_id|director_id|
+--------+-----------+
|       1|          1|
+--------+-----------+

